In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv("/content/train_data.txt", sep=":::", engine="python", names=["id", "title", "genre", "plot"])
print(train_df.head())

   id                               title       genre  \
0   1       Oscar et la dame rose (2009)       drama    
1   2                       Cupid (1997)    thriller    
2   3   Young, Wild and Wonderful (1980)       adult    
3   4              The Secret Sin (1915)       drama    
4   5             The Unrecovered (2007)       drama    

                                                plot  
0   Listening in to a conversation between his do...  
1   A brother and sister with a past incestuous r...  
2   As the bus empties the students for their fie...  
3   To help their unemployed father make ends mee...  
4   The film's title refers not only to the un-re...  


In [3]:
train_df["genre"] = train_df["genre"].str.strip()

In [4]:
print(train_df["genre"].unique())

['drama' 'thriller' 'adult' 'documentary' 'comedy' 'crime' 'reality-tv'
 'horror' 'sport' 'animation' 'action' 'fantasy' 'short' 'sci-fi' 'music'
 'adventure' 'talk-show' 'western' 'family' 'mystery' 'history' 'news'
 'biography' 'romance' 'game-show' 'musical' 'war']


In [5]:
train_df["clean_plot"] = train_df["plot"].str.lower()

In [6]:
print(train_df[["plot", "clean_plot"]].head(2))

                                                plot  \
0   Listening in to a conversation between his do...   
1   A brother and sister with a past incestuous r...   

                                          clean_plot  
0   listening in to a conversation between his do...  
1   a brother and sister with a past incestuous r...  


In [7]:
import re

def clean_text(text):
    text = re.sub(r"[^a-z\s]", " ", text)
    return text

train_df["clean_plot"] = train_df["clean_plot"].apply(clean_text)

In [8]:
print(train_df[["plot", "clean_plot"]].head(2))

                                                plot  \
0   Listening in to a conversation between his do...   
1   A brother and sister with a past incestuous r...   

                                          clean_plot  
0   listening in to a conversation between his do...  
1   a brother and sister with a past incestuous r...  


In [9]:
print(train_df["plot"].iloc[0])
print("---")
print(train_df["clean_plot"].iloc[0])

 Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few weeks to live. Furious, he refuses to speak to anyone except straight-talking Rose, the lady in pink he meets on the hospital stairs. As Christmas approaches, Rose uses her fantastical experiences as a professional wrestler, her imagination, wit and charm to allow Oscar to live life and love to the full, in the company of his friends Pop Corn, Einstein, Bacon and childhood sweetheart Peggy Blue.
---
 listening in to a conversation between his doctor and parents     year old oscar learns what nobody has the courage to tell him  he only has a few weeks to live  furious  he refuses to speak to anyone except straight talking rose  the lady in pink he meets on the hospital stairs  as christmas approaches  rose uses her fantastical experiences as a professional wrestler  her imagination  wit and charm to allow oscar to live life and love to the f

In [10]:
train_df["clean_plot"] = train_df["clean_plot"].apply(lambda t: re.sub(r"\s+", " ", t).strip())

In [11]:
print(train_df["clean_plot"].iloc[0])

listening in to a conversation between his doctor and parents year old oscar learns what nobody has the courage to tell him he only has a few weeks to live furious he refuses to speak to anyone except straight talking rose the lady in pink he meets on the hospital stairs as christmas approaches rose uses her fantastical experiences as a professional wrestler her imagination wit and charm to allow oscar to live life and love to the full in the company of his friends pop corn einstein bacon and childhood sweetheart peggy blue


In [12]:
test_df = pd.read_csv("/content/test_data_solution.txt", sep=":::", engine="python", names=["id", "title", "genre", "plot"])
test_df["genre"] = test_df["genre"].str.strip()
test_df["clean_plot"] = test_df["plot"].str.lower().apply(clean_text).apply(lambda t: re.sub(r"\s+", " ", t).strip())

print(test_df[["genre", "clean_plot"]].head(2))

      genre                                         clean_plot
0  thriller  l r brane loves his life his car his apartment...
1    comedy  spain march quico is a very naughty child of t...


In [13]:
blind_df = pd.read_csv("/content/test_data.txt", sep=":::", engine="python", names=["id", "title", "plot"])
blind_df["clean_plot"] = blind_df["plot"].str.lower().apply(clean_text).apply(lambda t: re.sub(r"\s+", " ", t).strip())

print(blind_df[["title", "clean_plot"]].head(2))

                        title  \
0       Edgar's Lunch (1998)    
1   La guerra de papá (1977)    

                                          clean_plot  
0  l r brane loves his life his car his apartment...  
1  spain march quico is a very naughty child of t...  


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000, stop_words="english", ngram_range=(1, 2))

X_train = vectorizer.fit_transform(train_df["clean_plot"])
X_test = vectorizer.transform(test_df["clean_plot"])
X_blind = vectorizer.transform(blind_df["clean_plot"])

y_train = train_df["genre"]
y_test = test_df["genre"]

print(X_train.shape)
print(X_test.shape)

(54214, 20000)
(54200, 20000)


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.5917158671586716


In [16]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
print("Naive Bayes accuracy:", accuracy_score(y_test, nb_pred))

svm_model = LinearSVC()
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
print("Linear SVM accuracy:", accuracy_score(y_test, svm_pred))

Naive Bayes accuracy: 0.5101845018450184
Linear SVM accuracy: 0.5775461254612546


In [17]:
blind_preds = model.predict(X_blind)
blind_df["predicted_genre"] = blind_preds
blind_df[["id", "title", "predicted_genre"]].to_csv("predictions.csv", index=False)

print(blind_df[["title", "predicted_genre"]].head(10))

                                          title predicted_genre
0                         Edgar's Lunch (1998)            drama
1                     La guerra de papá (1977)            drama
2                  Off the Beaten Track (2010)      documentary
3                       Meu Amigo Hindu (2015)            drama
4                            Er nu zhai (1955)            drama
5                           Riddle Room (2016)            drama
6                               L'amica (1969)            drama
7                         Ina Mina Dika (1989)           comedy
8   Equinox Special: Britain's Tornados (2005)      documentary
9                                 Press (2011)            drama


In [18]:
from google.colab import files
files.download("predictions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>